# Phase1: データ拡張で写真はどう変わる？

> ⏱ 所要時間: 説明を読む 8分 ＋ 実行 1分

## このNotebookのゴール
**データ拡張で、1枚の写真がどのように変わるのかを、自分の目で見て理解する。**

AIに学習させる前に、「拡張した写真を人間が見ても、同じアリ（ハチ）に見えるか」を確認しておくことが大切です。
ありえない写真（被写体が消える、まったく別物に見える等）を作る拡張は、AIを混乱させて**逆効果**になるからです。

## 前のNotebookとのつながり
Notebook 00 で、アリ・ハチの写真を train / val / test に分けました。ここでは train の写真1枚を使って、拡張を試します。

## このNotebookでやること
1. 10種類の拡張を、1枚の写真に**1つずつ**かけて、並べて見る
2. 拡張は**ランダム**で、同じ操作でも毎回結果が違うことを確認する
3. 拡張の**強さ**を変えると、見た目がどう変わるかを見る
4. 複数の拡張を**組み合わせる**（次のNotebook以降で使う形）

## キーワード
| 用語 | 意味 |
|---|---|
| **拡張（Augmentation）** | 写真を加工して、別の写真として使うこと |
| **`torchvision.transforms.v2`** | 拡張の部品がそろったPythonのライブラリ。以降 `v2` と書く |
| **幾何変換** | 形・位置を変える加工（反転・回転・切り抜き・拡大縮小など） |
| **色変換** | 明るさ・色合いを変える加工 |

In [ ]:
import sys
from pathlib import Path

# 正しい環境（プロジェクトの .venv）で動いているか確認する
try:
    import torch  # noqa: F401
except ModuleNotFoundError:
    raise SystemExit(
        f"torch が見つかりません。今のPython: {sys.executable}\n"
        "→ カーネルを「Python (image_augment_lab)」（または プロジェクトの .venv/bin/python）に切り替えてください。"
    )

# src/ をimportできるようにする（notebooks/ からでもプロジェクト直下からでも動く）
sys.path.insert(0, str(next(p for p in [Path.cwd() / "src", Path.cwd().parent / "src"] if p.exists())))
from nb_utils import *

%matplotlib inline

## 準備: 写真を1枚読み込む

アリの写真を1枚読み込んで、224×224ピクセルの大きさにそろえます（拡張の前の「元の写真」です）。
`random.seed(0); torch.manual_seed(0)` は、乱数の出方を固定する呪文です。**何度実行しても同じ結果になります**（再現のため）。

In [ ]:
import torch, random
from torchvision.transforms import v2   # 拡張の部品がそろったライブラリ

random.seed(0); torch.manual_seed(0)    # 乱数を固定（毎回同じ結果にするため）
files = sorted((DATA / "train" / "ants").glob("*.jpg"))   # アリの写真の一覧
img = v2.Resize((224, 224))(Image.open(files[0]).convert("RGB"))  # 1枚目を読み込み、サイズをそろえる
img

## 1. 拡張ギャラリー（10種類）

次のセルは、元の写真に**10種類の拡張を1つずつ**かけて並べます。
`src/visualize_augment.py` に定義済みの拡張を使います。

| 拡張 | 何をする？ | 狙い（AIにどうなってほしい？） |
|---|---|---|
| 水平反転 | 左右を裏返す | 左向き・右向きどちらでも分かるように |
| 垂直反転 | 上下を裏返す | 上下逆でも分かるように（ふつうの写真では不自然なことが多い） |
| 回転 | 少し傾ける | 傾いて撮られた写真にも対応 |
| 切り抜き+拡大 | 一部を切り出して元の大きさに拡大 | 位置や大きさが違っても分かるように |
| アフィン | 位置をずらす・拡大縮小 | 同上 |
| 色調 | 明るさ・彩度・色相を変える | 照明やカメラの違いに強く |
| グレースケール | 白黒にする | 色に頼りすぎないように |
| ぼかし | ピントを甘くする | ピンぼけにも対応 |
| 透視変換 | 斜めから見たように歪める | 撮影角度の違いに対応 |
| TrivialAugment | 上記のようなものをランダムに1つ選んで適用 | 自動で適当な拡張を選ぶ |

In [ ]:
import visualize_augment as V

# 「元画像」＋「10種類の拡張をかけた画像」を並べて表示
items = [("元画像", img)] + [(name, aug(img)) for name, aug in V.AUGMENTS.items()]
show_images([im for _, im in items], [n for n, _ in items], cols=4);

**見るポイント**（実際の画像を見ながら考えてみてください）

- **回転・アフィン・透視変換**: 画像の角が**黒く**なっていませんか？ AIが「黒い部分」を手がかりにしてしまう恐れがあります。
- **水平反転**: 右下の撮影ID（UGA…）の文字が**鏡文字**になっています。こういう「写真の中の文字」はアリ/ハチと無関係なので、手がかりにされると困ります。
- **色調**: 背景の青が大きく変わっています。**アリとハチを見分けるのに色は必要**でしょうか？（→ Notebook 04 で実験します）
- **すべての拡張で**、「人間が見てもアリ」に見えますか？ 見えない拡張は、使ってはいけません。

## 2. 拡張は「ランダム」 — 同じ操作でも毎回違う

拡張には**ランダム性**があります。たとえば「ランダム切り抜き」は、切り出す場所と大きさを**毎回ランダムに決めます**。

学習では、エポック（全写真を1周する単位）ごとに拡張をやり直します。
つまり、**同じ1枚の写真が、エポックのたびに少しずつ違う写真としてAIに見せられます**。
60エポック学習すれば、1枚の写真が**60通りの見え方**になります。これが「写真が増える」の正体です。

次のセルで、同じ「ランダム切り抜き」を**8回**かけてみます。毎回違う場所が切り出されます。

In [ ]:
# scale=(0.4, 1.0) は「元の写真の40%〜100%の面積をランダムに切り出す」という意味
aug = v2.RandomResizedCrop(224, scale=(0.4, 1.0))

# 同じ画像に8回かけて、結果が毎回違うことを確認する
show_images([aug(img) for _ in range(8)], [f"{i+1}回目" for i in range(8)], cols=4);

## 3. 拡張の「強さ」を変えてみる

多くの拡張には**強さ**のパラメータがあります（回転なら何度まで回すか、など）。

- **弱すぎる**: ほとんど変わらず、拡張の意味がない
- **強すぎる**: 被写体が欠けたり、不自然になったりして、AIが学べない

**基本は「弱めから試す」**です。次のセルは、4種類の拡張について、強さを4段階に変えた結果を並べます（上から1行ずつ）。

In [ ]:
def strength_row(title, make, levels):
    # levels の値を1つずつ使って拡張を作り、同じ写真にかけて横に並べる
    ims = [make(l)(img) for l in levels]
    show_images(ims, [f"{title}={l}" for l in levels], cols=len(levels), size=2.6)

# 回転: 度数を 10→90 に強くする（(d, d) は「ちょうどd度」の意味）
strength_row("回転degrees", lambda d: v2.RandomRotation(degrees=(d, d)), [10, 30, 60, 90])
# 切り抜き: 切り出す面積の割合。小さいほど「強く切り取る」（0.8=80%残す、0.08=8%だけ残す）
strength_row("切り抜きscale下限", lambda s: v2.RandomResizedCrop(224, scale=(s, s)), [0.8, 0.5, 0.2, 0.08])
# 色調: 明るさ・コントラスト・彩度をどれだけ変えるか
strength_row("色調の強さ", lambda k: v2.ColorJitter(k, k, k, min(k, 0.5)), [0.1, 0.3, 0.6, 0.9])
# ぼかし: sigmaが大きいほど強くぼける
strength_row("ぼかしsigma", lambda s: v2.GaussianBlur(9, sigma=(s, s)), [0.5, 1.5, 3.0, 5.0]);

**見るポイント**
- 各行の**右端**（一番強い設定）は、アリに見えますか？ たとえば切り抜き `0.08` は、アリがほとんど写っていないはずです。
  → 強すぎると、「アリの写真」というラベルが**嘘**になってしまいます（ラベルノイズ）。
- 逆に左端は、元の写真とほとんど変わりません。
- ちょうどよい強さは、**「人間が見てもそのクラスと分かる範囲で、できるだけ変化がある」**あたりです。

## 4. 拡張を組み合わせる（`Compose`）

実際の学習では、複数の拡張を**順番に**かけます。`v2.Compose([...])` は「リストの順に拡張を実行する」部品です。

このあとの実験で使う **`basic`** は、**切り抜き → 左右反転 → 色調変更**の3つを組み合わせたものです。
1枚の写真に8回かけると、毎回違う写真ができます。

In [ ]:
basic = v2.Compose(T.AUG_SETS["basic"])    # train.py に定義済みの「basic」拡張の組み合わせ
original = Image.open(files[0]).convert("RGB")
show_images([basic(original) for _ in range(8)], [f"basic {i+1}" for i in range(8)], cols=4);

## 🧪 試してみよう

### 演習1: 強さを変えて、限界を見つける
**やること**: 上の「3. 強さを変えてみる」のセルで、回転の行の **`[10, 30, 60, 90]`** を **`[5, 15, 45, 180]`** に書き換えて再実行します。
- **180度**にすると写真が上下逆さまになります。アリの写真として自然でしょうか？
- **分かること**: 拡張には「**ちょうどよい強さ**」があり、極端に強いと不自然になります。この感覚が、次のNotebookで拡張を選ぶ土台になります。

### 演習2: 別の写真で試す（ハチの写真）
**やること**: 準備のセルの **`"ants"`** を **`"bees"`** に書き換え、このNotebookを**上から**実行し直します。
- アリでは気にならなかった拡張が、ハチでは不自然に見えたりしないか？ 色調変更の影響は違って見えますか？

### 演習3: 自分で考える（コードは不要）
もし分類する対象が、**「手書きの数字の6と9」**だったら、使ってはいけない拡張は何でしょう？
- ヒント: 6を180度回転すると何に見える？ → **回転**と**上下反転**は危険。
- **分かること**: 「どの拡張が安全か」は、**課題によって違う**。「ラベルが変わらない拡張だけを使う」のが鉄則です。

---

## ✅ まとめ

- 拡張は、写真を加工して「別の見え方の写真」を作る。
- 拡張は**ランダム**で、**エポックごとに違う写真**ができる。だから1枚が何通りにも使える。
- **強さ**は「人間が見ても同じクラスと分かる範囲」にとどめる。**弱めから試す**。
- 拡張後の写真は、**必ず自分の目で確認する**。
- 「ラベルが変わらない（アリがハチに見えない）」拡張だけを使う。

➡ 次は **`02_baseline.ipynb`**。拡張を使わずにAIを学習させ、**過学習**を実際に見ます（拡張の効果を測る**基準**を作ります）。